In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure output directories exist
os.makedirs('reports', exist_ok=True)

print(" Initializing Day 6 Advanced Analytics Engine...")

# ==========================================
# SETUP SIMULATED DATA ENVIRONMENT
# ==========================================
np.random.seed(101)
dates = pd.date_range(start="2024-01-01", end="2026-06-01", freq="B")
funds = [f"Fund_{i}" for i in range(1, 41)]

# Generate daily returns for the 40 funds
return_data = {fund: np.random.normal(0.0004, 0.012, len(dates)) for fund in funds}
df_returns = pd.DataFrame(return_data, index=dates)

# ==========================================
# TASK 1: VALUE AT RISK (VaR) & CONDITIONAL VaR (CVaR)
# ==========================================
var_cvar_list = []
for fund in funds:
    fund_rets = df_returns[fund]
    # 5th percentile for 95% Historical VaR
    var_95 = np.percentile(fund_rets, 5)
    # Mean of returns below the VaR threshold
    cvar_95 = fund_rets[fund_rets <= var_95].mean()
    
    var_cvar_list.append({
        "Fund": fund,
        "VaR_95": var_95,
        "CVaR_95": cvar_95
    })

df_var_report = pd.DataFrame(var_cvar_list)
df_var_report.to_csv("var_cvar_report.csv", index=False)
print(" Task 1 Complete: Saved var_cvar_report.csv")

# ==========================================
# TASK 2: ROLLING 90-DAY SHARPE RATIO (TOP 5 FUNDS)
# ==========================================
top_5_funds = funds[:5]
Rf_daily = 0.065 / 252

plt.figure(figsize=(12, 6))
for fund in top_5_funds:
    rolling_mean = df_returns[fund].rolling(90).mean()
    rolling_std = df_returns[fund].rolling(90).std()
    rolling_sharpe = (rolling_mean - Rf_daily) / rolling_std * np.sqrt(252)
    plt.plot(rolling_sharpe.index, rolling_sharpe, label=f"{fund} Rolling Sharpe")

plt.title("Task 2 — Rolling 90-Day Sharpe Ratio Over Time (Top 5 Funds)", fontsize=12, fontweight='bold')
plt.xlabel("Timeline")
plt.ylabel("Annualized Sharpe Ratio")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.savefig("reports/rolling_sharpe_chart.png", dpi=300, bbox_inches='tight')
plt.close()
print(" Task 2 Complete: Saved reports/rolling_sharpe_chart.png")

# ==========================================
# TASK 3: INVESTOR COHORT ANALYSIS
# ==========================================
# Simulating an investor profile dataset
num_investors = 500
cohort_data = {
    "Investor_ID": [f"INV_{i:04d}" for i in range(1, num_investors + 1)],
    "First_Tx_Year": np.random.choice([2024, 2025], size=num_investors, p=[0.45, 0.55]),
    "SIP_Amount": np.random.uniform(2000, 25000, size=num_investors),
    "Total_Invested": np.random.uniform(50000, 500000, size=num_investors),
    "Fund_Preference": np.random.choice(["Equity Bluechip", "Midcap Growth", "Smallcap Index", "Balanced Debt"], size=num_investors)
}
df_investors = pd.DataFrame(cohort_data)

df_cohort = df_investors.groupby("First_Tx_Year").agg(
    Avg_SIP_Amount=("SIP_Amount", "mean"),
    Total_Capital_Invested=("Total_Invested", "sum"),
    Top_Preferred_Fund=("Fund_Preference", lambda x: x.mode()[0])
).reset_index()

df_cohort.to_csv("cohort_analysis.csv", index=False)
print(" Task 3 Complete: Saved cohort_analysis.csv")

# ==========================================
# TASK 4: SIP CONTINUATION & RETENTION ANALYSIS
# ==========================================
# Simulate a sequence of transactions for continuity evaluation
tx_list = []
for i in range(1, 150): # 149 sample active recurring profiles
    n_tx = np.random.randint(4, 12) # generate varying transaction lengths
    base_dates = pd.date_range(start="2025-01-01", periods=n_tx, freq="30D")
    # inject unexpected random processing/deposit delays to test logic limits
    noise = np.random.choice([0, 2, 5, 38], size=n_tx, p=[0.7, 0.15, 0.1, 0.05])
    actual_dates = base_dates + pd.to_timedelta(noise, unit='D')
    
    for dt in actual_dates:
        tx_list.append({"Investor_ID": f"INV_{i:04d}", "Tx_Date": dt})

df_tx = pd.DataFrame(tx_list).sort_values(["Investor_ID", "Tx_Date"])
df_tx['Prev_Tx_Date'] = df_tx.groupby("Investor_ID")['Tx_Date'].shift(1)
df_tx['Gap_Days'] = (df_tx['Tx_Date'] - df_tx['Prev_Tx_Date']).dt.days

# Filter users who have completed 6 or more transaction entries
investor_counts = df_tx['Investor_ID'].value_counts()
eligible_investors = investor_counts[investor_counts >= 6].index

df_continuity = df_tx[df_tx['Investor_ID'].isin(eligible_investors)].groupby("Investor_ID").agg(
    Avg_Transaction_Gap=("Gap_Days", "mean"),
    Max_Transaction_Gap=("Gap_Days", "max")
).reset_index()

df_continuity['Status'] = np.where(df_continuity['Max_Transaction_Gap'] > 35, "at-risk", "active")
df_continuity.to_csv("sip_continuity.csv", index=False)
print(" Task 4 Complete: Saved sip_continuity.csv")

# ==========================================
# TASK 6: SECTOR CONCENTRATION ANALYSIS (HHI INDEX)
# ==========================================
sectors = ["Technology", "Financial Services", "Healthcare", "Energy", "Consumer Goods"]
hhi_list = []

for fund in funds:
    # Generate random portfolio weights summing to 1.0
    raw_weights = np.random.dirichlet(np.ones(len(sectors)), size=1)[0]
    # Herfindahl-Hirschman Index computation
    hhi_score = np.sum(raw_weights ** 2)
    
    hhi_list.append({
        "Fund": fund,
        "HHI_Index": hhi_score,
        "Concentration_Risk": "High" if hhi_score > 0.25 else ("Moderate" if hhi_score > 0.15 else "Low")
    })

df_hhi = pd.DataFrame(hhi_list)
df_hhi.to_csv("sector_hhi.csv", index=False)

# Render concentration summary charts
plt.figure(figsize=(10, 5))
df_hhi.set_index("Fund")["HHI_Index"].head(15).plot(kind='bar', color='darkcyan')
plt.axhline(y=0.25, color='r', linestyle='--', label='High Concentration Bound (0.25)')
plt.title("Task 6 — Portfolio Sector Concentration Risk Metrics (HHI Index)", fontsize=11, fontweight='bold')
plt.ylabel("Computed HHI Score")
plt.legend()
plt.savefig("reports/sector_concentration_chart.png", dpi=300, bbox_inches='tight')
plt.close()
print(" Task 6 Complete: Saved sector_hhi.csv and reports/sector_concentration_chart.png")

print("\n ==========================================")
print("TASK 7: ADVANCED ANALYTICS STRATEGIC SUMMARY")
print("==============================================")
highest_var_fund = df_var_report.sort_values(by="VaR_95").iloc[0]['Fund']
worst_cvar_val = df_var_report.sort_values(by="CVaR_95").iloc[0]['CVaR_95']
most_active_cohort = df_cohort.sort_values(by="Total_Capital_Invested", ascending=False).iloc[0]['First_Tx_Year']
at_risk_count = (df_continuity['Status'] == 'at-risk').sum()
highest_hhi_fund = df_hhi.sort_values(by="HHI_Index", ascending=False).iloc[0]['Fund']

print(f"1. Tail-Risk Volatility Profile: {highest_var_fund} exhibits the absolute sharpest historical 95% Value-at-Risk threshold variance.")
print(f"2. Deep Drawdown Exposure: The worst conditional loss distribution (CVaR) caps downside moves at {worst_cvar_val:.4%}.")
print(f"3. Capital Intake Dominance: Structural cohort modeling identifies the {most_active_cohort} investor block as your primary macro capitalization source.")
print(f"4. Retention Alert Metrics: Transaction spacing triggers flagged a total of {at_risk_count} institutional recurring accounts as 'at-risk' (>35 days pause).")
print(f"5. Asset Concentration Risk: Structural auditing maps out {highest_hhi_fund} as the least diversified sector basket via HHI indices.")

 Initializing Day 6 Advanced Analytics Engine...
 Task 1 Complete: Saved var_cvar_report.csv
 Task 2 Complete: Saved reports/rolling_sharpe_chart.png
 Task 3 Complete: Saved cohort_analysis.csv
 Task 4 Complete: Saved sip_continuity.csv
 Task 6 Complete: Saved sector_hhi.csv and reports/sector_concentration_chart.png

TASK 7: ADVANCED ANALYTICS STRATEGIC SUMMARY
1. Tail-Risk Volatility Profile: Fund_27 exhibits the absolute sharpest historical 95% Value-at-Risk threshold variance.
2. Deep Drawdown Exposure: The worst conditional loss distribution (CVaR) caps downside moves at -2.6320%.
3. Capital Intake Dominance: Structural cohort modeling identifies the 2025 investor block as your primary macro capitalization source.
4. Retention Alert Metrics: Transaction spacing triggers flagged a total of 39 institutional recurring accounts as 'at-risk' (>35 days pause).
5. Asset Concentration Risk: Structural auditing maps out Fund_20 as the least diversified sector basket via HHI indices.
